In [ ]:
import pandas as pd
import requests
import time

countries = ["USA","CHN","IND","JPN","PAK","EMU", "arab","WLD"]

indicators = {
    "NY.GDP.MKTP.KD.ZG":"GDP_Growth_Percent",
    "FP.CPI.TOTL.ZG":"Inflation_Percent",
    "SP.POP.TOTL":"Population_Total",
    "GC.DOD.TOTL.GD.ZS":"Debt_to_GDP_Percent"
}

start_year = 2000
end_year = 2026

all_data = []

for country in countries:
    for code, name in indicators.items():
        url = f"https://api.worldbank.org/v2/country/{country}/indicator/{code}?date={start_year}:{end_year}&format=json&per_page=1000"
        
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            data = r.json()[1]

            for entry in data:
                all_data.append({
                    "Year": entry["date"],
                    "Country": country,
                    "Indicator": name,
                    "Value": entry["value"]
                })

            time.sleep(1)  # avoid rate limits

        except Exception as e:
            print(f"Error for {country}-{name}: {e}")

df = pd.DataFrame(all_data)

df = df.pivot_table(
    index=["Year","Country"],
    columns="Indicator",
    values="Value"
).reset_index()

df["Data_Type"] = "Historical"
df["Source"] = "World Bank"

df.to_csv("global_macro_panel_2000_2026.csv", index=False)

print("✅ Dataset created successfully")


Error for CHN-Population_Total: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=20)
Error for IND-Population_Total: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=20)
Error for JPN-Inflation_Percent: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=20)
Error for arab-GDP_Growth_Percent: list index out of range
Error for arab-Inflation_Percent: list index out of range
Error for arab-Population_Total: list index out of range
✅ Dataset created successfully


In [5]:
import pandas as pd
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ===============================
# CONFIGURATION
# ===============================

countries = {
    "USA": "United States",
    "CHN": "China",
    "IND": "India",
    "JPN": "Japan",
    "PAK": "Pakistan",
    "EMU": "Euro Area",
    "ARB": "Arab World",
    "WLD": "World"
}

indicators = {
    "NY.GDP.MKTP.KD.ZG": "GDP_Growth_Percent",
    "FP.CPI.TOTL.ZG": "Inflation_Percent",
    "SP.POP.TOTL": "Population_Total",
    "GC.DOD.TOTL.GD.ZS":"Debt_to_GDP_Percent"
}

start_year = 2000
end_year = 2026   # World Bank historical limit

# ===============================
# SESSION WITH RETRY LOGIC
# ===============================

session = requests.Session()

retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)

# ===============================
# DATA COLLECTION
# ===============================

all_data = []

for iso, country_name in countries.items():
    print(f"Downloading data for {country_name}...")

    for code, indicator_name in indicators.items():

        url = (
            f"https://api.worldbank.org/v2/country/{iso}/indicator/{code}"
            f"?date={start_year}:{end_year}&format=json&per_page=1000"
        )

        try:
            response = session.get(url, timeout=60)
            response.raise_for_status()
            json_data = response.json()

            # Check if valid data returned
            if len(json_data) < 2:
                print(f"⚠ No data found for {country_name} - {indicator_name}")
                continue

            for entry in json_data[1]:
                if entry["value"] is not None:
                    all_data.append({
                        "Year": int(entry["date"]),
                        "Country": country_name,
                        "ISO3": iso,
                        "Indicator": indicator_name,
                        "Value": entry["value"]
                    })

            time.sleep(0.5)  # avoid rate limit

        except Exception as e:
            print(f"❌ Error for {country_name} - {indicator_name}: {e}")

# ===============================
# CREATE DATAFRAME
# ===============================

df = pd.DataFrame(all_data)

# Pivot to panel format
df = df.pivot_table(
    index=["Year", "Country", "ISO3"],
    columns="Indicator",
    values="Value"
).reset_index()

# Sort properly
df = df.sort_values(["Country", "Year"])

# Add metadata columns
df["Data_Type"] = "Historical"
df["Source"] = "World Bank API"

# ===============================
# SAVE FILE
# ===============================

output_file = "global_macro_panel_2000_2023.csv"
df.to_csv(output_file, index=False)

print("\n✅ Dataset created successfully!")
print(f"📁 File saved as: {output_file}")
print(f"📊 Total rows: {len(df)}")



✅ Dataset created successfully!
📁 File saved as: global_macro_panel_2000_2023.csv
📊 Total rows: 200


In [ ]:
import pandas as pd
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ===============================
# CONFIGURATION
# ===============================

indicators = {
    "NY.GDP.MKTP.KD.ZG": "•	Population_Growth_Rate",
    "FP.CPI.TOTL.ZG": "GDP_Per_Capita"
}

start_year = 2000
end_year = 2026   # Latest available historical year

# ===============================
# SESSION WITH RETRY LOGIC
# ===============================

session = requests.Session()

retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)

# ===============================
# STEP 1: GET ALL COUNTRIES
# ===============================

print("Downloading country list...")

countries_url = "https://api.worldbank.org/v2/country?per_page=400&format=json"
response = session.get(countries_url)
country_json = response.json()

countries = {}

for c in country_json[1]:
    # Keep only actual countries (exclude aggregates like World, Arab World, etc.)
    if c["region"]["value"] != "Aggregates":
        countries[c["id"]] = c["name"]

print(f"✅ Total countries found: {len(countries)}")

# ===============================
# STEP 2: DOWNLOAD INDICATOR DATA
# ===============================

all_data = []

for iso, country_name in countries.items():
    print(f"Downloading {country_name}...")

    for code, indicator_name in indicators.items():

        url = (
            f"https://api.worldbank.org/v2/country/{iso}/indicator/{code}"
            f"?date={start_year}:{end_year}&format=json&per_page=1000"
        )

        try:
            response = session.get(url, timeout=60)
            response.raise_for_status()
            json_data = response.json()

            if len(json_data) < 2:
                continue

            for entry in json_data[1]:
                if entry["value"] is not None:
                    all_data.append({
                        "Year": int(entry["date"]),
                        "Country": country_name,
                        "ISO3": iso,
                        "Indicator": indicator_name,
                        "Value": entry["value"]
                    })

            time.sleep(0.3)

        except Exception as e:
            print(f"❌ Error for {country_name} - {indicator_name}: {e}")

# ===============================
# STEP 3: CREATE PANEL DATA
# ===============================

df = pd.DataFrame(all_data)

df = df.pivot_table(
    index=["Year", "Country", "ISO3"],
    columns="Indicator",
    values="Value"
).reset_index()

df = df.sort_values(["Country", "Year"])

# Metadata
df["Data_Type"] = "Historical"
df["Source"] = "World Bank API"

# ===============================
# STEP 4: SAVE FILE
# ===============================

output_file = "global_all_countries_macro_2000_2026.csv"
df.to_csv(output_file, index=False)

print("\n✅ Dataset created successfully!")
print(f"📁 File saved as: {output_file}")
print(f"🌍 Total rows: {len(df)}")
print(f"🏳 Total countries: {df['Country'].nunique()}")


✅ Total countries found: 217

✅ Dataset created successfully!
📁 File saved as: global_all_countries_macro_2000_2026.csv
🌍 Total rows: 5204
🏳 Total countries: 214
